In [1]:
%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain_chroma 
%pip install -Uq langchain langchain-community langchain-openai 
%pip install -Uq python_dotenvpip

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement python_dotenvpip (from versions: none)
ERROR: No matching distribution found for python_dotenvpip


In [3]:
pip install --upgrade pip

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.1.1
    Uninstalling pip-25.1.1:
      Successfully uninstalled pip-25.1.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

c:\Users\selvi\PycharmProjects\Multi_Modal_RAG_System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "./docs/prachis_organics_KB.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


📄 Partitioning document: ./docs/prachis_organics_KB.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
No languages specified, defaulting to English.
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 2456.34it/s]


✅ Extracted 257 elements


In [6]:

elements

In [7]:
# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [8]:
elements[36].to_dict()

{'type': 'Image',
 'element_id': '91f9664dce11b9b497fdc65ec23d5a1d',
 'text': 'BE',
 'metadata': {'coordinates': {'points': ((np.float64(909.4884166666666),
     np.float64(1488.667784722222)),
    (np.float64(909.4884166666666), np.float64(2301.6886180555553)),
    (np.float64(1594.9050833333333), np.float64(2301.6886180555553)),
    (np.float64(1594.9050833333333), np.float64(1488.667784722222))),
   'system': 'PixelSpace',
   'layout_width': 2898,
   'layout_height': 4094},
  'last_modified': '2026-03-31T14:14:31',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './docs',
  'filename': 'prachis_organics_KB.pdf'}}

In [9]:
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

Found 6 images


{'type': 'Image',
 'element_id': '84d59831562c68a3e4a85a36c78e68af',
 'text': 'Pr i |',
 'metadata': {'coordinates': {'points': ((np.float64(909.4884166666666),
     np.float64(2528.829284722222)),
    (np.float64(909.4884166666666), np.float64(3469.454284722222)),
    (np.float64(1850.1134166666666), np.float64(3469.454284722222)),
    (np.float64(1850.1134166666666), np.float64(2528.829284722222))),
   'system': 'PixelSpace',
   'layout_width': 2898,
   'layout_height': 4094},
  'last_modified': '2026-03-31T14:14:31',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './docs',
  'filename': 'prachis_organics_KB.pdf'}}

In [10]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html 

Found 2 tables


{'type': 'Table',
 'element_id': 'd35e69db22e8d786ba0425fce00d356b',
 'text': 'Category Categories of Recipients',
 'metadata': {'detection_class_prob': 0.5966160893440247,
  'is_extracted': 'partial',
  'coordinates': {'points': ((np.float64(344.2892150878906),
     np.float64(2804.56689453125)),
    (np.float64(344.2892150878906), np.float64(3069.4609375)),
    (np.float64(2527.65771484375), np.float64(3069.4609375)),
    (np.float64(2527.65771484375), np.float64(2804.56689453125))),
   'system': 'PixelSpace',
   'layout_width': 2898,
   'layout_height': 4094},
  'last_modified': '2026-03-31T14:14:31',
  'text_as_html': '<table><tbody><tr><td>Category</td><td>Categories of Recipients</td></tr></tbody></table>',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 10,
  'file_directory': './docs',
  'filename': 'prachis_organics_KB.pdf',
  'parent_id': '06de5a4c5c25dbcc4825c88e4c46f44c'}}

In [6]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 29 chunks


In [12]:
# View all chunks
chunks



In [13]:
# All unique types
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>"}

In [14]:
# View a single chunk
chunks[2].to_dict()


{'type': 'CompositeElement',
 'element_id': 'ec894e76-1565-490b-925e-682545d645d8',
 'text': 'Product type: Body Care/Body Soap\n\nProduct use-case: - Wet your body and lather the soap between your palms or on a loofah.\n\n- Massage gently onto your skin in circular motions.\n\n- Rinse thoroughly with water and pat dry.\n\n- Follow with your favorite body lotion or oil for a lasting glow.\n\nPricing (divide this into the variants respectively):\n\n● 100gm: ₹199 (Discounted from ₹249)\n\nProduct description (if any):\n\nExperience the timeless beauty ritual of Ayurveda with our handcrafted Ubtan Soap, enriched with powerful herbs like Sandalwood, Amba Haldi, Nagarmotha, Khus, Rose, and Bawachi. This traditional blend deeply cleanses, gently exfoliates with Almond Powder, and helps brighten dull, uneven skin. Infused with pure essential oils, it soothes, refreshes, and\n\nleaves your skin feeling smooth, radiant, and naturally glowing after every wash. Ideal For: All skin types seeking n

In [15]:
# View original elements
chunks[11].metadata.orig_elements[-1].to_dict()
# Note: 4th chunk has the first image + 11th chunk has the first table in the sample PDF

{'type': 'NarrativeText',
 'element_id': '86cd200e2346facc800d3b994eec7458',
 'text': 'As of the Effective Date of this Privacy Policy, we do not have actual knowledge that we “share” or “sell” (as those terms are defined in applicable law) personal information of individuals under 16 years of age.',
 'metadata': {'detection_class_prob': 0.9038650393486023,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(344.1282653808594),
     np.float64(1040.2089218669091)),
    (np.float64(344.1282653808594), np.float64(1235.1043444710758)),
    (np.float64(2430.1071661779206), np.float64(1235.1043444710758)),
    (np.float64(2430.1071661779206), np.float64(1040.2089218669091))),
   'system': 'PixelSpace',
   'layout_width': 2898,
   'layout_height': 4094},
  'last_modified': '2026-03-31T14:14:31',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 12,
  'file_directory': './docs',
  'filename': 'prachis_organics_KB.pdf',
  'parent_id': '3856981aa8bd947ec3c

In [ ]:
    def separate_content_types(chunk):
        """Analyze what types of content are in a chunk"""
        content_data = {
            'text': chunk.text,
            'tables': [],
            'images': [],
            'types': ['text']
        }
        
        # Check for tables and images in original elements
        if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
            for element in chunk.metadata.orig_elements:
                element_type = type(element).__name__
                
                # Handle tables
                if element_type == 'Table':
                    content_data['types'].append('table')
                    table_html = getattr(element.metadata, 'text_as_html', element.text)
                    content_data['tables'].append(table_html)
                
                # Handle images
                elif element_type == 'Image':
                    if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                        content_data['types'].append('image')
                        content_data['images'].append(element.metadata.image_base64)
        
        content_data['types'] = list(set(content_data['types']))
        return content_data

In [8]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """
        
        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
        
                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed  
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [9]:
def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/29
     Types found: ['text', 'image']
     Tables: 0, Images: 3
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: **Searchable Description for Document Content Retrieval:**

**Prachi’s Organics KB**

**Website:** prachisorganics.com

**Products Overview:**

1. **Aloe Vera & Rosemary Hair Shampoo**
   - **Type:** ...
   Processing chunk 2/29
     Types found: ['text', 'image']
     Tables: 0, Images: 2
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: **Product 4: Orange Face Wash**

- **Type:** Skin Care/Face Wash
- **Use-Case:** Suitable for all skin types, including sensitive and combination skin. Targets dull, uneven, or tired-looking skin. Pro...
   Processing chunk 3/29
     Types found: ['text', 'image']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
  

In [19]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "Prachi\\u2019s Organics KB\\n\\nWebsite: prachisorganics.com\\n\\nProducts:\\n\\n\\ud83c\\udf6fProduct 1\\n\\nProduct name: Aloe Vera & Rosemary Hair Shampoo - 200ml Product type: Hair Care/Hair Shampoo Product use-case: Reduces dandruff, promotes healthy hair growth, soothes dry scalp Pricing (divide this into the variants respectively):\\n\\nPricing (divide this into the variants respectively):\\n\\n\\u25cf Default Title: \\u20b9699 (Discounted from \\u20b9599)\\n\\nProduct description (if any):\\n\\nA gentle, plant-powered shampoo that cleanses without stripping natural oils. Formulated with mild cleansers to remove buildup while maintaining scalp balance. Aloe Vera and Glycerin hydrate and soften hair, while Rosemary and Lemongrass Essential Oils invigorate the scalp, promote healthier growth, and add natural shine. Leaves hair feeling fresh, lightweight, and beautifully nourished with every wash. Ideal For: All hair types, esp

In [20]:
def export_chunks_to_json(chunks, filename="chunks_export1.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 29 chunks to chunks_export1.json


In [ ]:
def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")
        
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")
    
    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv1/chroma_db


In [15]:
# After your retrieval
query = "In how many days can i get my refund?"
retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

NameError: name 'export_chunks_to_json' is not defined

In [ ]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)
    
    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv3/chroma_db")
    
    print("🎉 Pipeline completed successfully!")
    return db

# Run the complete pipeline

In [16]:
db = run_complete_ingestion_pipeline("./docs/prachis_organics_KB.pdf")

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


🚀 Starting RAG Ingestion Pipeline
📄 Partitioning document: ./docs/prachis_organics_KB.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
No languages specified, defaulting to English.
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


✅ Extracted 257 elements
🔨 Creating smart chunks...
✅ Created 29 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/29
     Types found: ['text', 'image']
     Tables: 0, Images: 3
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: **Searchable Description for Document Content Retrieval:**

**Prachi’s Organics KB**

**Website:** prachisorganics.com

**Products:**

1. **Aloe Vera & Rosemary Hair Shampoo**
   - **Type:** Hair Care...
   Processing chunk 2/29
     Types found: ['text', 'image']
     Tables: 0, Images: 2
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: **Document Content Description for Searchability:**

**Product 4: Orange Face Wash**
- **Type:** Skin Care/Face Wash
- **Use-Case:** Suitable for all skin types, including sensitive and combination sk...
   Processing chunk 3/29
     Types found: ['text', 'image']
    

In [31]:
# Query the vector store
query = "Kitne din mein delivery milegi?"

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer in hinglish if the query is in hinglish. Otherwise, provide the answer in English. Use the text, tables, and images above to inform your response. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)

Agar aapka order prepaid hai ya Cash on Delivery (COD) hai, toh dispatch hone mein 5 business din lagenge. Metro cities mein delivery hone mein 5 se 7 business din lagenge aur remote areas ya villages mein 10 se 12 business din lagenge. Peak seasons, natural calamities, ya unforeseen courier issues ki wajah se delivery mein delay ho sakta hai.
